# Experiment 001 — Typo Robustness: Colab Pilot

Full pilot pipeline in six cells:
**clone → install → build items → build dictionary → generate → analyze + download**

Scoring is performed inline during generation (no separate scoring step).
Uses `qwen_1b5_pilot` (Qwen2.5-1.5B-Instruct, ungated, no HF auth needed) on a Colab T4.
The pilot config (`configs/pilot.yaml`) sets `is_confirmatory: false`, so pinned model
revisions are not enforced.

**Before starting:** Runtime → Change runtime type → T4 GPU.

**Run cells top to bottom, in order, in one sitting.** State (the cloned repo, the
working directory, the `run_streaming` helper defined in Cell 1) persists across
cells within a runtime, but not across a runtime restart — if the runtime restarts
or disconnects, start again from Cell 1. (Cell 5's generation step is itself
resumable and will pick up where it left off; the environment setup in Cells 1–2
is not, and needs to be redone after a restart.)

---
### Pipeline overview

| Cell | Tool | Time (approx) | Output |
|------|------|---------------|--------|
| 1 | clone + CPU deps | 2 min | environment ready |
| 2 | GPU stack + spaCy transformer | 5 min | CUDA verified |
| 3 | `build_task_items` + `build_annotated_dataset` | 5–10 min | `data/items/*.jsonl` |
| 4 | `build_dictionary` (SCOWL) | 1 min | `data/wordlists/en_us_pinned.txt` |
| 5 | `run_generation` | 30–60 min | `results/pilot/pilot_generations.jsonl` |
| 6 | `run_analysis` + `build_report` + download | 5 min | `pilot_results.zip` |

Every cell streams its subprocess's stdout and stderr live as it runs — including
progress bars — instead of only showing output once the process exits
(`subprocess.run` without this buffers by default, which silently hid crashes for
most of a run in earlier sessions). If a cell's process fails, its exit code and
its full stdout and stderr (separately) are printed in a clearly labeled block
right before the traceback — the same information a
`try: ... except subprocess.CalledProcessError as e: print(e.returncode, e.stdout, e.stderr)`
would give you, but without losing the live progress bar to get it.

---
### Resuming after an interruption (Cell 5)

Cell 5's generation is idempotent and flushes every row to disk the instant it is
generated (not batched), so if the Colab session disconnects, crashes (e.g. a CUDA
out-of-memory event), or you interrupt it: **just re-run Cell 5.** It skips every
row already written and resumes exactly where it left off; at most the handful of
requests in flight at that moment are regenerated, never a whole batch.

---
### Key outputs to review after Cell 6

- **Discordant rate per cell** → sets N for the main run (Connor 1987; design/06 §6.3)
- **Clean accuracy A₀** → validate against expected GSM-Symbolic / MMLU bands
- **`max_new_tokens`** → set to 99th percentile of clean-correct generation lengths
- **Coverage report** in `exclusions.jsonl` → how many items were perturbed successfully per condition
- **Extraction tier distribution** → what fraction of answers were parsed at each tier
- **Regime C reasoning coverage** → check `instance params validated` in Cell 3 output; items that failed regex extraction appear in `exclusions.jsonl`

### Cell 1 — Clone repo and install CPU dependencies

In [ ]:
import subprocess, sys, threading

def run_streaming(cmd, **kwargs):
    # Combine stderr into stdout at the OS level
    process = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, **kwargs
    )

    combined_chunks: list[bytes] = []

    # Single stream loop: highly efficient and keeps order as the OS flushes it
    while True:
        chunk = process.stdout.read(1024)
        if not chunk:
            break
        
        combined_chunks.append(chunk)
        sys.stdout.write(chunk.decode("utf-8", errors="replace"))
        sys.stdout.flush()

    process.wait()
    combined_text = b"".join(combined_chunks).decode("utf-8", errors="replace")

    if process.returncode != 0:
        banner = "=" * 72
        rule = "-" * 72
        # Print the error report to stderr so calling environments know it failed
        print(f"\n{banner}", file=sys.stderr)
        print(f"[run_streaming] FAILED: {cmd}", file=sys.stderr)
        print(f"[run_streaming] return code: {process.returncode}", file=sys.stderr)
        print(f"{rule}\nCOMBINED OUTPUT:\n{combined_text}", file=sys.stderr)
        print(banner, file=sys.stderr)
        
        raise subprocess.CalledProcessError(
            process.returncode, cmd, output=combined_text, stderr=combined_text
        )

run_streaming(["git", "clone", "https://github.com/natSegOS/glamor-research-onboarding.git"])

%cd glamor-research-onboarding/experiments/001_typo_robustness

run_streaming([sys.executable, "-u", "-m", "pip", "install", "-e", ".", "-q"])
run_streaming([sys.executable, "-u", "-m", "pip", "install", "-r", "requirements.txt", "-q"])
print("CPU deps installed")

### Cell 2 — Install GPU stack and verify CUDA

`spacy-transformers` is installed here because it uses PyTorch and runs GPU-accelerated
on the T4 during the annotation step (Cell 3).  The `en_core_web_trf` model is the
RoBERTa-based spaCy pipeline used for both pilot and confirmatory annotation runs —
the pilot must be methodologically identical to the main study (design/11 §11.2).

In [ ]:
import subprocess, sys

run_streaming([sys.executable, "-u", "-m", "pip", "install", "-r", "requirements-gpu.txt", "-q"])

# Download the spaCy transformer model (GPU-accelerated via spacy-transformers).
# en_core_web_trf uses RoBERTa-base; published benchmarks at spacy.io/models/en.
# The exact model SHA is recorded in data/items/annotation_PROVENANCE.json.
run_streaming([sys.executable, "-u", "-m", "spacy", "download", "en_core_web_trf", "-q"])

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("No GPU found — check Runtime > Change runtime type > T4 GPU")

### Cell 3 — Fetch task items and annotate with K_P(x) key terms

**`build_task_items`** downloads 100 items per dataset from HuggingFace and writes
pinned JSONL files to `data/items/`.  The `p1` GSM-Symbolic variant is used because
it is derived from 100 annotated templates in Apple's companion GitHub repo
(github.com/apple/ml-gsm-symbolic), which this cell also clones.

The Apple template repo provides the symbolic **structure** (parameter names, types,
answer formula); the HF question text provides the **instance values**.  The tool
matches the template's format string against each HF question to extract the real
parameter values, then validates `answer_function(**extracted) == gold_answer`.
Items that pass are fully Regime C capable; items that fail are excluded gracefully
and appear in `exclusions.jsonl`.  Check the `instance params validated` line in
the cell output to see the actual coverage rate.

**`build_annotated_dataset`** annotates each item with its frozen K_P(x) key-term
set using `en_core_web_trf` (design/04 §4.6).  GPU-accelerated on the T4.

In [ ]:
import subprocess, sys

# Clone Apple's GSM-Symbolic template repo.
# The templates/ and generated_data/ directories are needed to enrich the
# HF items with question_annotated fields for Regime C operand-swap.
# License: Apple custom open-source license (source/binary redistribution
# permitted); generated_data/ is CC-BY-NC-ND-4.0 (used locally, not committed).
run_streaming([
    "git", "clone", "--depth", "1",
    "https://github.com/apple/ml-gsm-symbolic.git",
    "/tmp/ml-gsm-symbolic",
])

# Fetch 100 items per dataset from HuggingFace, enrich GSM-Symbolic items
# with question_annotated and validated instance parameters from the Apple
# template repo.
run_streaming([
    sys.executable, "-u", "tools/build_task_items.py",
    "--reasoning-items",   "100",
    "--mcq-items",         "100",
    "--gsm-config",        "p1",
    "--seed",              "1729",
    "--output-directory",  "data/items",
    "--gsm-templates-dir", "/tmp/ml-gsm-symbolic",
])

# Annotate with frozen K_P(x) key terms using en_core_web_trf.
# GPU-accelerated via spacy-transformers; takes ~5 min on a T4.
run_streaming([
    sys.executable, "-u", "tools/build_annotated_dataset.py",
    "--model-name", "en_core_web_trf",
    "--items-dir",  "data/items",
    "--force",
])

print("Items annotated and ready in data/items/")

### Cell 4 — Build the SCOWL English dictionary

Downloads SCOWL 2020.12.07 (Kevin Atkinson, wordlist.aspell.net) from its
SourceForge release archive and builds `data/wordlists/en_us_pinned.txt`.

The vocabulary is restricted to the **`words`** sub-category only (english +
american dialect, size band ≤60) — not the full bundle SCOWL's own `mk-list`
tool would pull in by default (`abbreviations`, `upper`, `proper-names`,
`contractions`, and the `special` category of hacker jargon and roman
numerals). Those extra categories exist so a *spell-checker* doesn't flag
"Mr.", "TCP", or "IV" as misspelled — a different question from what
`is_word` needs to answer (is this edited token a real word a reader would
recognize as distinct and meaningful — the check that separates Regime A
from Regime B). Measured effect of the full bundle: it counts 100% of single
letters, 51.8% of two-letter strings, and 7.3% of three-letter strings as
"real words" (mostly lower-cased abbreviations and roman numerals), which
inflates false "landed on a real word" rejections when constructing Regime A
items. The `words`-only list drops that to 11.2% / 3.8% for two/three-letter
strings while remaining exactly as citable and reproducible (still the sole
source, still one dialect, still SHA-pinned in `PROVENANCE.json`).

This dictionary is the `is_word` predicate that separates Regime A nonword
typos from Regime B real-word shifts. See `data/wordlists/README.md` for the
full rationale.

In [ ]:
import subprocess, sys

# Download the prebuilt SCOWL 2020.12.07 release from SourceForge (the
# built final/ word lists; NOT the en-wl/wordlist GitHub repo, which is
# SCOWL source + a Makefile with no prebuilt final/ directory).
run_streaming([
    "wget", "-q", "-O", "/tmp/scowl.tar.gz",
    "https://sourceforge.net/projects/wordlist/files/SCOWL/2020.12.07/scowl-2020.12.07.tar.gz/download",
])
run_streaming(["tar", "-xzf", "/tmp/scowl.tar.gz", "-C", "/tmp/"])

# Build the pinned vocabulary from the size-60 'words' lists only (english +
# american dialect) — deliberately narrower than SCOWL's own mk-list default,
# which also pulls in abbreviations/proper-names/upper/hacker-jargon/roman-
# numerals. See tools/build_dictionary.py and data/wordlists/README.md for
# why: those categories are not "real words" in the sense is_word needs.
run_streaming([
    sys.executable, "-u", "tools/build_dictionary.py",
    "--scowl-path",     "/tmp/scowl-2020.12.07/final/",
    "--scowl-max-size", "60",
    "--scowl-dialect",  "american",
])

print("Dictionary ready — data/wordlists/en_us_pinned.txt")

### Cell 5 — Generate pilot outputs

Runs all conditions in `configs/pilot.yaml` (Regimes A, B, C) against
`qwen_1b5_pilot` (Qwen2.5-1.5B-Instruct). Scoring is performed inline: each
generation is scored the instant it is produced (four-way parse-status
classifier: VALID / UNPARSEABLE / CLARIFICATION / REFUSAL) — no separate
scoring step.

**Per-row streaming, not batching.** The runner drives vLLM's engine directly
(`add_request` + `step`) and writes+flushes each row to disk the moment that
one request finishes decoding — not after a fixed-size batch of requests all
complete. If the runtime disconnects, the GPU runs out of memory, or you
interrupt the cell, **just re-run this cell**: it skips every row already on
disk and resumes from there, losing at most the handful of requests that were
actually in flight — never a whole batch's worth of generations (this is
exactly what happened in earlier sessions before this fix: a mid-run crash
meant zero rows made it past the batch boundary).

**Parallelism on a single T4:** vLLM's own scheduler already keeps the GPU
saturated via continuous batching — every pending request is submitted up
front and the engine decides concurrency, so there is nothing to configure
here. `--shard-index`/`--shard-count` (see `tools/run_generation.py --help`)
exist for splitting one config's work across *multiple GPUs/sessions* (e.g.
the USC cluster) — not useful on Colab's single T4, where running two worker
processes would just make them compete for the same 16 GB of VRAM.

Expected time: ~30–60 min on a T4.

In [ ]:
import subprocess, sys

git_commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], capture_output=True, text=True
).stdout.strip() or "unpinned"

run_streaming([
    sys.executable, "-u", "tools/run_generation.py",
    "--config",           "configs/pilot.yaml",
    "--model",            "qwen_1b5_pilot",
    "--output-directory", "results/pilot",
    "--dictionary",       "data/wordlists/en_us_pinned.txt",
    "--git-commit",       git_commit,
])

print("Generation done — results/pilot/pilot_generations.jsonl")

### Cell 6 — Analyze results and download

**`run_analysis`** produces the per-cell accuracy table, discordant-pair rates,
mediation estimates, and figures in `analysis/pilot/`.

**`build_report`** writes a self-contained HTML drill-down at
`results/pilot/report.html` with global statistics and a per-item diff view.

Everything is zipped and downloaded as `pilot_results.zip`.

In [ ]:
import subprocess, sys, zipfile, pathlib

generations_path = "results/pilot/pilot_generations.jsonl"

# Statistical summary: cell table, discordant rates, mediation, figures.
run_streaming([
    sys.executable, "-u", "tools/run_analysis.py",
    "--generations",      generations_path,
    "--output-directory", "analysis/pilot",
])

# Self-contained HTML drill-down report.
run_streaming([
    sys.executable, "-u", "tools/build_report.py",
    "--generations", generations_path,
    "--output",      "results/pilot/report.html",
])

# Zip all outputs and download.
zip_path = pathlib.Path("pilot_results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in pathlib.Path("analysis/pilot").rglob("*"):
        if path.is_file():
            zf.write(path)
    for extra in ["results/pilot/report.html",
                  "results/pilot/pilot_generations.jsonl",
                  "results/pilot/pilot_exclusions.jsonl"]:
        if pathlib.Path(extra).exists():
            zf.write(extra)

try:
    from google.colab import files
    files.download(str(zip_path))
    print("Downloaded pilot_results.zip")
except ImportError:
    print(f"Results at {zip_path.resolve()}")